# core

> Claude code api backend for fastllm

In [ ]:
#| default_exp core

In [ ]:
import asyncio, re
from pathlib import Path
from claude_agent_sdk import (query, ClaudeAgentOptions, create_sdk_mcp_server, tool,
    AssistantMessage, ToolUseBlock, StreamEvent, ResultMessage)
from fastllm.types import *
from fastllm.anthropic import (norm_sse_event, norm_tool_calls, norm_parts,
    norm_finish, norm_usage, finalize_usage, denorm_msgs, delta_index_fn, cost)
from fastllm.streaming import mk_acollect_stream
from fastspec.errors import APIError
from llmsurgery.ant import *

In [ ]:
MCP_SERVER_NAME = "fastllm"
WORK_DIR = Path.home() / ".fastllm-claude-agent"
WORK_DIR.mkdir(exist_ok=True)

In [ ]:
sess_dir(WORK_DIR)

Path('/Users/jhoward/.claude/projects/-Users-jhoward--fastllm-claude-agent')

In [ ]:
SERVER_TOOLS = ["WebSearch", "WebFetch"]

def msgs_to_recs(msgs, model="claude-sonnet-4-6", session_id=None):
    "Convert fastllm Msgs to CC session records, with ids derived stably from the content."
    den = denorm_msgs(msgs)
    sid = session_id or stable_uuid("fastllm-claude-code:" + canon(den))
    for d in den:
        content = d.get("content", [])
        if isinstance(content, list):
            for b in content:
                if not isinstance(b, dict): continue
                b.pop("cache_control", None)
                if b.get("type") == "tool_use":
                    nm = b.get("name", "")
                    if nm and not nm.startswith("mcp__") and nm not in SERVER_TOOLS: b["name"] = f"{MCP_PREFIX}{nm}"
            if d["role"] == "user" and all(isinstance(b, dict) and b.get("type") == "text" for b in content):
                content = "".join(b.get("text", "") for b in content)
        d["content"] = content
    return sid, msgs2recs(den, key=sid, cwd=WORK_DIR, model=model, entrypoint="sdk-py")

In [ ]:
def mk_stub(name, desc, schema, block):
    @tool(name, desc, schema)
    async def _stub(args): await block.wait()
    return _stub

In [ ]:
def _last_user_text(m):
    "Extract text from a user Msg's content parts."
    return "\n".join(p.text or '' for p in m.content if p.type == PartType.text)

def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build prompt + options for Claude Code SDK query. Last msg is the new turn; rest is resumed history."
    system, tools = kwargs.get('system'), kwargs.get('tools')
    if msgs and msgs[-1].role == 'user' and _last_user_text(msgs[-1]):
        *history, last = msgs
        prompt = _last_user_text(last)
    else:
        history, prompt = msgs, "." # continue tool results, works fine but if it becomes an issue make tool use prompt
    
    block = asyncio.Event()
    mcp_tools, allowed = [], []
    for t in (tools or []):
        nm, desc, params = fn_schema(t)
        if nm:
            mcp_tools.append(mk_stub(nm, desc or "", params, block))
            allowed.append(f"mcp__{MCP_SERVER_NAME}__{nm}")
    mcp_servers = {MCP_SERVER_NAME: create_sdk_mcp_server(MCP_SERVER_NAME, tools=mcp_tools)} if mcp_tools else {}
    cc_tools = ["WebSearch", "WebFetch"] if kwargs.get('web_search_options') is not None else []

    opt_kw = dict(model=model, env={'ANTHROPIC_API_KEY': ''}, cwd=str(WORK_DIR), include_partial_messages=True,
        permission_mode="default", system_prompt=system or "", mcp_servers=mcp_servers, allowed_tools=allowed,
        strict_mcp_config=True, tools=cc_tools)

    if history:
        sid, recs = msgs_to_recs(history, model=model)
        save_sess(recs, sid, WORK_DIR)
        opt_kw['resume'] = sid
    opts = ClaudeAgentOptions(**opt_kw)
    return dict(prompt=prompt, options=opts, block=block)

In [ ]:
MCP_PREFIX = f"mcp__{MCP_SERVER_NAME}__"

async def claude_acollect_stream(payload, **kwargs):
    opts, prompt = payload["options"], payload["prompt"]
    async def _gen():
        gen = query(prompt=prompt, options=opts)
        saw_tool = False
        base, max_in_msg = 0, -1          # global index offset across messages
        try:
            async for msg in gen:
                if isinstance(msg, ResultMessage) and msg.is_error:
                    txt = msg.result or "; ".join(msg.errors or []) or msg.subtype
                    m = re.search(r'\b(\d{3})\b', txt or '')
                    raise APIError(txt, provider='claude_code', model=opts.model,
                        status_code=int(m.group(1)) if m else None, raw=msg)
                if isinstance(msg, AssistantMessage):
                    if any(isinstance(b, ToolUseBlock) and b.name.startswith(MCP_PREFIX) for b in (msg.content or [])): saw_tool = True
                elif isinstance(msg, StreamEvent):
                    ev = msg.event
                    t = ev.get("type")
                    if t == "message_start":
                        base += max_in_msg + 1      # advance past prev message's blocks
                        max_in_msg = -1
                    elif t in ("content_block_start","content_block_delta","content_block_stop") and "index" in ev:
                        i = ev["index"]
                        max_in_msg = max(max_in_msg, i)
                        ev = {**ev, "index": i + base}
                    if t == "message_stop" and saw_tool: return
                    cb = ev.get("content_block", {})
                    if cb.get("type") == "tool_use" and cb.get("name", "").startswith(MCP_PREFIX):
                        ev = {**ev, "content_block": {**cb, "name": cb["name"][len(MCP_PREFIX):]}}
                    delta = norm_sse_event(ev)
                    for tc in (delta.tool_calls or []):
                        if tc.name in ["WebSearch", "WebFetch"]: tc.server = True
                    yield delta
        finally:
            try: await gen.aclose()
            except Exception: pass
    async for o in mk_acollect_stream(_gen(), index_fn=delta_index_fn, api_name='claude_code', **kwargs): yield o

In [ ]:
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream, cost=cost)

### Tests

In [ ]:
from fastllm.chat import mk_msgs, acomplete, lite_mk_func, AsyncChat

In [ ]:
msgs = mk_msgs("What is 2+2?")
r = await acomplete(msgs, 'claude-sonnet-4-6', api_name='claude_code', stream=True)
async for o in r:
    if isinstance(o, Completion): print(f"\n--- finish: {o.finish_reason}, usage: {o.usage}")
    elif t := o.get('text'): print(t, end='')

4
--- finish: stop, usage: Usage(prompt_tokens=1086, completion_tokens=5, total_tokens=1091, cached_tokens=1083, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 3, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 1083, 'output_tokens': 5, 'output_tokens_details': {'thinking_tokens': 0}, 'iterations': [{'input_tokens': 3, 'output_tokens': 5, 'cache_read_input_tokens': 1083, 'cache_creation_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}]})


In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

In [ ]:
msgs = mk_msgs("What is 3+5 and 10+5? Use the simple_add tool in parallel.")
r = await acomplete(msgs, 'claude-sonnet-4-6', api_name='claude_code', stream=True, tools=[lite_mk_func(simple_add)])

async for o in r:
    if isinstance(o, Completion):
        print(f"\n--- finish: {o.finish_reason}")
        print(f"--- tool_calls: {o.tool_calls}")
    elif isinstance(o, Part): print(f"\n[Part {o.type}: {o.data}]")
    elif t := o.get('text'): print(t, end='')

Sure! I'll calculate both sums simultaneously!
[Part tool_use: {'caller': {'type': 'direct'}, 'id': 'toolu_015kSH7Fr8AUwjSS4zVvohKb', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}]

[Part tool_use: {'caller': {'type': 'direct'}, 'id': 'toolu_013f8kdtgQCqnQQ8D1aoBWCs', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 5}, 'server': False}]

--- finish: tool_calls
--- tool_calls: [ToolCall(id='toolu_015kSH7Fr8AUwjSS4zVvohKb', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}), ToolCall(id='toolu_013f8kdtgQCqnQQ8D1aoBWCs', name='simple_add', arguments={'a': 10, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}})]


In [ ]:
def delta_text(o):
    "Extract printable content from streaming delta, return None if nothing to print"
    if isinstance(o, Part) and o.type == PartType.tool_result:  return f'🔧 {o.data['name']}\n'
    if isinstance(o,dict): 
        if o.get('thinking'):    return '🧠'
        elif txt:=o.get('text'): return txt
    return None

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', tools=[simple_add])
res = await chat("What is 7+3? Use the tool.", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠Sure!🔧 simple_add
🧠🧠🧠**7 + 3 = 10**

That's the complete result from the `simple_add` tool. Nothing further needed!

In [ ]:
def multiply(a: int, b: int) -> int:
    "Multiply two numbers"
    return a * b

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', tools=[simple_add, multiply])
res = await chat("Calculate 3+5 and 4*6 in parallel using tools.", stream=True, max_steps=5)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠Sure! Since these are independent calculations, I'll run both tools simultaneously:🔧 simple_add
🔧 multiply
🧠🧠🧠Here are the results from both parallel calculations:

- **3 + 5 = 8**
- **4 × 6 = 24**

In [ ]:
res = await chat("What was the last result.", stream=True, max_steps=5)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠The last result was **4 × 6 = 24**.

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', search='l')
res = await chat("Can you search the web for weather in Istanbul", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠🔧 WebSearch
Here's the weather outlook for Istanbul in **July 2026**:

### 🌡️ Temperature
- **Highs:** 28–33°C (82–91°F)
- **Lows:** 17–23°C (63–73°F)

### ☀️ Sunshine & Humidity
- About **12 hours of sunshine** per day (~79% of daylight hours)
- Humidity around **65%** — actually the driest/least humid month of the year

### 🌧️ Rainfall
- Very little rain — roughly **4 rainy days** and only ~26mm (1 inch) for the whole month

### 🌊 Sea Temperature
- Around **23°C (73°F)** — warm enough for swimming

**Overall:** Hot, sunny, and mostly dry — classic peak summer. It's also peak tourist season, so expect crowds.

Sources:
- [Istanbul July Weather – AccuWeather](https://www.accuweather.com/en/tr/istanbul/318251/july-weather/318251)
- [Istanbul Weather in July 2026 – weather25.com](https://www.weather25.com/europe/turkey/istanbul?page=month&month=July)
- [Istanbul Weather in July – EaseWeather](https://www.easeweather.com/europe/turkey/istanbul/july)
- [Istanbul Weather in July – WhereAn

In [ ]:
res = await chat("What is the weather like again? Just tell me from previous the response", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠Here's the Istanbul weather summary from the previous search:

- **Highs:** 28–33°C (82–91°F), **Lows:** 17–23°C (63–73°F)
- **Sunshine:** ~12 hours/day, humidity around 65%
- **Rainfall:** Very little — ~4 rainy days and 26mm for the whole month
- **Sea Temperature:** ~23°C (73°F)

Overall: Hot, sunny, and mostly dry — classic peak summer.

Sources:
- [Istanbul July Weather – AccuWeather](https://www.accuweather.com/en/tr/istanbul/318251/july-weather/318251)
- [Istanbul Weather in July 2026 – weather25.com](https://www.weather25.com/europe/turkey/istanbul?page=month&month=July)
- [Istanbul Weather in July – EaseWeather](https://www.easeweather.com/europe/turkey/istanbul/july)
- [Istanbul Weather in July – WhereAndWhen](https://www.whereandwhen.net/when/middle-east/turkey/istanbul/july/)

In [ ]:
chat.use

total=3,300 | in=2,991 | out=281 | cached=83.7% | cache_new=484 | reasoning=28 | $0.0005 | claude-sonnet-4-6

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()